# Futures Term Structure: Contango, Backwardation, and Roll Yield

## Learning Objectives

By completing this notebook, you will:
1. Understand futures term structure and its information content
2. Identify contango and backwardation market regimes
3. Calculate and interpret roll yield
4. Build term structure models for commodity forecasting
5. Extract forward curve features for prediction

## Prerequisites
- Futures contract basics
- Data retrieval pipeline (Module 2.1)
- Time series concepts

## Estimated Time: 80 minutes

---

## 1. Setup and Imports

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Data and modeling
import yfinance as yf
from scipy import interpolate, optimize
from sklearn.decomposition import PCA

# Configuration
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Environment ready!")

## 2. What is Term Structure?

**Term structure** (forward curve) = Prices of futures contracts with different maturities

**Key concepts:**

1. **Spot price** ($S_t$): Current cash market price
2. **Futures price** ($F_{t,T}$): Price today for delivery at time $T$
3. **Forward curve**: Plot of $F_{t,T}$ vs $T$ at fixed time $t$

**Market regimes:**

1. **Contango**: $F_{t,T_2} > F_{t,T_1}$ for $T_2 > T_1$ (upward sloping)
   - Far contracts more expensive than near
   - Typical for storable commodities (carry cost)
   - **Negative roll yield** (long positions lose money rolling forward)

2. **Backwardation**: $F_{t,T_2} < F_{t,T_1}$ for $T_2 > T_1$ (downward sloping)
   - Near contracts more expensive than far
   - Indicates supply shortage or high convenience yield
   - **Positive roll yield** (long positions gain from rolling)

**Why it matters:**
- Forward curve contains market expectations
- Roll yield affects investment returns
- Term structure shape indicates supply/demand balance

## 3. Simulating Term Structure Data

For pedagogy, let's first simulate term structure with known characteristics.

In [ ]:
def simulate_term_structure(spot, maturities, regime='contango', 
                           slope=0.02, noise_std=0.5):
    """
    Simulate futures term structure.
    
    Parameters
    ----------
    spot : float
        Spot price
    maturities : array
        Time to maturity in years
    regime : str
        'contango' or 'backwardation'
    slope : float
        Steepness of curve
    noise_std : float
        Random noise level
    
    Returns
    -------
    array
        Futures prices
    """
    if regime == 'contango':
        # Upward sloping (storage cost dominates)
        futures = spot * np.exp(slope * maturities)
    elif regime == 'backwardation':
        # Downward sloping (convenience yield dominates)
        futures = spot * np.exp(-slope * maturities)
    else:
        raise ValueError("regime must be 'contango' or 'backwardation'")
    
    # Add noise
    noise = np.random.normal(0, noise_std, len(maturities))
    futures = futures + noise
    
    return futures

# Simulate both regimes
spot_price = 75.0
maturities = np.array([0.083, 0.167, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0])  # In years
maturity_labels = ['1M', '2M', '3M', '6M', '9M', '1Y', '1.5Y', '2Y']

contango_curve = simulate_term_structure(spot_price, maturities, 'contango', slope=0.05)
backwardation_curve = simulate_term_structure(spot_price, maturities, 'backwardation', slope=0.08)

print("Simulated term structures")
print(f"Spot price: ${spot_price:.2f}")

In [ ]:
# Visualize both regimes
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Contango
axes[0].plot(maturities, contango_curve, 'o-', linewidth=2, markersize=8, color='red')
axes[0].axhline(spot_price, color='blue', linestyle='--', linewidth=2, label='Spot Price')
axes[0].set_xlabel('Time to Maturity (years)', fontsize=12)
axes[0].set_ylabel('Futures Price ($)', fontsize=12)
axes[0].set_title('Contango (Upward Sloping)', fontsize=14, fontweight='bold')
axes[0].set_xticks(maturities)
axes[0].set_xticklabels(maturity_labels)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Backwardation
axes[1].plot(maturities, backwardation_curve, 'o-', linewidth=2, markersize=8, color='green')
axes[1].axhline(spot_price, color='blue', linestyle='--', linewidth=2, label='Spot Price')
axes[1].set_xlabel('Time to Maturity (years)', fontsize=12)
axes[1].set_ylabel('Futures Price ($)', fontsize=12)
axes[1].set_title('Backwardation (Downward Sloping)', fontsize=14, fontweight='bold')
axes[1].set_xticks(maturities)
axes[1].set_xticklabels(maturity_labels)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Real Futures Data: Building the Forward Curve

Real futures data requires mapping contract codes to maturities.

In [ ]:
# Fetch multiple crude oil futures contracts
# Yahoo Finance convention: CLF24.NYM = Feb 2024, CLG24.NYM = Mar 2024, etc.
# Month codes: F=Jan, G=Feb, H=Mar, J=Apr, K=May, M=Jun, N=Jul, Q=Aug, U=Sep, V=Oct, X=Nov, Z=Dec

# For simplicity, use generic continuous contracts which represent rolled positions
# In production, you'd query actual contract prices by expiration

# Let's simulate a more realistic term structure using recent WTI data
wti_spot = yf.download('CL=F', start='2023-01-01', end='2024-01-01', progress=False)['Adj Close']

# Get recent price as "spot"
recent_spot = wti_spot.iloc[-1]

print(f"Recent WTI spot: ${recent_spot:.2f}")
print("\nNote: In practice, you would fetch actual futures contracts by expiration.")
print("For this example, we'll construct synthetic curves based on historical patterns.")

In [ ]:
# Construct realistic term structures based on typical oil market patterns
# We'll show both historical regimes

maturities_months = np.array([1, 2, 3, 6, 9, 12, 18, 24])  # Months
maturities_years = maturities_months / 12

# Scenario 1: Mild contango (typical for crude when storage is available)
contango_realistic = recent_spot * (1 + 0.01 * maturities_months**0.7)
contango_realistic += np.random.normal(0, 0.3, len(maturities_months))

# Scenario 2: Backwardation (supply disruption or strong demand)
backwardation_realistic = recent_spot * (1 - 0.015 * maturities_months**0.6)
backwardation_realistic += np.random.normal(0, 0.3, len(maturities_months))

# Create DataFrame
term_structure_df = pd.DataFrame({
    'maturity_months': maturities_months,
    'maturity_years': maturities_years,
    'contango': contango_realistic,
    'backwardation': backwardation_realistic
})

print("Term Structure Data:")
print(term_structure_df)

In [ ]:
# Visualize realistic curves
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(term_structure_df['maturity_months'], term_structure_df['contango'], 
        'o-', linewidth=2, markersize=8, color='red', label='Contango Scenario')
ax.plot(term_structure_df['maturity_months'], term_structure_df['backwardation'], 
        'o-', linewidth=2, markersize=8, color='green', label='Backwardation Scenario')
ax.axhline(recent_spot, color='blue', linestyle='--', linewidth=2, 
          label=f'Front Month (${recent_spot:.2f})')

ax.set_xlabel('Maturity (months)', fontsize=12)
ax.set_ylabel('Futures Price ($)', fontsize=12)
ax.set_title('Crude Oil Futures: Term Structure Scenarios', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Roll Yield: The Hidden Return Component

**Roll yield** = Return from holding futures as they converge to spot

**Formula:**
$$\text{Roll Yield} \approx \frac{F_{t,T_1} - F_{t,T_2}}{F_{t,T_2}} \times \frac{1}{T_2 - T_1}$$

where $T_1 < T_2$ (rolling from near to far contract)

**Intuition:**
- **Contango**: Far > Near → Pay up when rolling → Negative roll yield
- **Backwardation**: Far < Near → Benefit from rolling → Positive roll yield

**Importance:** Major component of commodity futures returns (can dominate spot price changes)

In [ ]:
def calculate_roll_yield(near_price, far_price, time_diff):
    """
    Calculate annualized roll yield.
    
    Parameters
    ----------
    near_price : float
        Near contract price
    far_price : float
        Far contract price
    time_diff : float
        Time between contracts (in years)
    
    Returns
    -------
    float
        Annualized roll yield (fraction)
    """
    return (near_price - far_price) / far_price / time_diff

# Calculate roll yield for both scenarios
# Roll from 1-month to 2-month contract

# Contango scenario
near_contango = term_structure_df.loc[0, 'contango']
far_contango = term_structure_df.loc[1, 'contango']
roll_yield_contango = calculate_roll_yield(near_contango, far_contango, 1/12)

# Backwardation scenario
near_back = term_structure_df.loc[0, 'backwardation']
far_back = term_structure_df.loc[1, 'backwardation']
roll_yield_back = calculate_roll_yield(near_back, far_back, 1/12)

print("Roll Yield Analysis (1M to 2M roll):")
print("="*60)
print(f"\nContango Scenario:")
print(f"  1-month price: ${near_contango:.2f}")
print(f"  2-month price: ${far_contango:.2f}")
print(f"  Roll yield: {roll_yield_contango*100:.2f}% (annualized)")
print(f"  → NEGATIVE (lose money rolling forward)")

print(f"\nBackwardation Scenario:")
print(f"  1-month price: ${near_back:.2f}")
print(f"  2-month price: ${far_back:.2f}")
print(f"  Roll yield: {roll_yield_back*100:.2f}% (annualized)")
print(f"  → POSITIVE (gain money rolling forward)")

In [ ]:
# Calculate roll yield across the entire curve
def compute_curve_roll_yields(prices, maturities):
    """
    Compute roll yield between adjacent contracts.
    """
    roll_yields = []
    maturity_pairs = []
    
    for i in range(len(prices) - 1):
        near = prices[i]
        far = prices[i + 1]
        time_diff = maturities[i + 1] - maturities[i]
        
        ry = calculate_roll_yield(near, far, time_diff)
        roll_yields.append(ry)
        maturity_pairs.append(f"{int(maturities[i]*12)}M-{int(maturities[i+1]*12)}M")
    
    return roll_yields, maturity_pairs

ry_contango, pairs = compute_curve_roll_yields(
    term_structure_df['contango'].values,
    term_structure_df['maturity_years'].values
)

ry_back, _ = compute_curve_roll_yields(
    term_structure_df['backwardation'].values,
    term_structure_df['maturity_years'].values
)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(pairs))
width = 0.35

ax.bar(x - width/2, np.array(ry_contango)*100, width, label='Contango', 
       color='red', alpha=0.7)
ax.bar(x + width/2, np.array(ry_back)*100, width, label='Backwardation', 
       color='green', alpha=0.7)

ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Contract Roll', fontsize=12)
ax.set_ylabel('Annualized Roll Yield (%)', fontsize=12)
ax.set_title('Roll Yield Across the Curve', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(pairs, rotation=45)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Term Structure Features for Forecasting

Extract quantitative features from the forward curve for modeling.

In [ ]:
def extract_term_structure_features(prices, maturities):
    """
    Extract statistical features from term structure.
    
    Parameters
    ----------
    prices : array
        Futures prices
    maturities : array
        Time to maturity
    
    Returns
    -------
    dict
        Feature dictionary
    """
    features = {}
    
    # 1. Slope (near vs far)
    features['slope_1m_12m'] = (prices[-4] - prices[0]) / prices[0]  # 1M to 12M
    
    # 2. Curvature (butterfly)
    features['curvature'] = prices[3] - 0.5 * (prices[0] + prices[-1])
    
    # 3. Level (average price)
    features['level'] = np.mean(prices)
    
    # 4. Spread (front vs back)
    features['front_back_spread'] = prices[0] - prices[-1]
    
    # 5. Calendar spread (adjacent contracts)
    features['calendar_spread_1_2'] = prices[1] - prices[0]
    
    # 6. Regime indicator
    features['regime'] = 'contango' if prices[-1] > prices[0] else 'backwardation'
    
    # 7. Average roll yield
    roll_yields, _ = compute_curve_roll_yields(prices, maturities)
    features['avg_roll_yield'] = np.mean(roll_yields)
    
    return features

# Extract features for both scenarios
features_contango = extract_term_structure_features(
    term_structure_df['contango'].values,
    term_structure_df['maturity_years'].values
)

features_back = extract_term_structure_features(
    term_structure_df['backwardation'].values,
    term_structure_df['maturity_years'].values
)

print("Term Structure Features:")
print("="*60)
print("\nContango Scenario:")
for k, v in features_contango.items():
    print(f"  {k}: {v}")

print("\nBackwardation Scenario:")
for k, v in features_back.items():
    print(f"  {k}: {v}")

## 7. PCA of Term Structure (Factor Analysis)

Term structure movements can be decomposed into:
1. **Level** (parallel shifts)
2. **Slope** (steepening/flattening)
3. **Curvature** (butterfly)

Use PCA to extract these factors from historical curves.

In [ ]:
# Simulate time series of term structures
np.random.seed(42)
n_days = 100

# Generate curves with varying regimes
curves = []
for t in range(n_days):
    spot_t = recent_spot + np.random.normal(0, 5)
    slope_t = np.random.uniform(-0.03, 0.03)
    regime = 'contango' if slope_t > 0 else 'backwardation'
    
    curve_t = simulate_term_structure(
        spot_t, 
        maturities_years, 
        regime=regime,
        slope=abs(slope_t),
        noise_std=0.3
    )
    curves.append(curve_t)

curves_array = np.array(curves)  # Shape: (n_days, n_contracts)

print(f"Generated {n_days} daily term structures")
print(f"Curve shape: {curves_array.shape}")

In [ ]:
# Apply PCA
pca = PCA(n_components=3)
principal_components = pca.fit_transform(curves_array)

# Variance explained
var_explained = pca.explained_variance_ratio_

print("PCA Results:")
print(f"  PC1 (Level) explains: {var_explained[0]*100:.1f}%")
print(f"  PC2 (Slope) explains: {var_explained[1]*100:.1f}%")
print(f"  PC3 (Curvature) explains: {var_explained[2]*100:.1f}%")
print(f"  Total: {var_explained.sum()*100:.1f}%")

In [ ]:
# Visualize principal components
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

component_names = ['Level', 'Slope', 'Curvature']

for i, ax in enumerate(axes):
    ax.plot(maturities_months, pca.components_[i], 'o-', linewidth=2, markersize=8)
    ax.axhline(0, color='black', linewidth=1, linestyle='--')
    ax.set_xlabel('Maturity (months)', fontsize=12)
    ax.set_ylabel('Loading', fontsize=12)
    ax.set_title(f'PC{i+1}: {component_names[i]}\n({var_explained[i]*100:.1f}% variance)', 
                fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretation:**
- **PC1 (Level)**: All contracts move together (parallel shift)
- **PC2 (Slope)**: Near vs far contracts move oppositely (steepening/flattening)
- **PC3 (Curvature)**: Middle contracts move differently from ends (butterfly)

---

## Exercise 1: Regime Classification

**Task:** Build a classifier to identify contango vs backwardation.

1. Generate 200 synthetic curves (100 each regime)
2. Extract term structure features
3. Train logistic regression classifier
4. Evaluate accuracy on test set

**Question:** Which features are most predictive?

In [ ]:
# YOUR CODE HERE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# YOUR CODE: Generate data, extract features, train classifier
classifier = None  # Replace with trained model

In [ ]:
# AUTO-GRADED TEST
def test_exercise_1():
    """Verify regime classification."""
    assert classifier is not None, "Train a classifier"
    print("✅ Exercise 1 complete! Check classification accuracy.")

# test_exercise_1()  # Uncomment after completing

---

## Exercise 2: Roll Yield Portfolio Strategy

**Task:** Simulate returns from a roll-yield-aware strategy.

**Strategy:**
- Go long when backwardation (positive roll yield expected)
- Stay flat or short when contango (negative roll yield)

1. Generate time series of regimes (alternating randomly)
2. Simulate spot returns and roll yields
3. Compute total return for: (a) buy-and-hold, (b) roll-yield strategy
4. Plot cumulative returns

**Expected:** Strategy outperforms in backwardation, underperforms in contango

In [ ]:
# YOUR CODE HERE

# YOUR CODE: Simulate strategy returns
buy_hold_returns = None  # Replace with your results
strategy_returns = None

---

## Exercise 3: Interpolating Forward Curves

**Task:** Create smooth forward curves from discrete contract prices.

1. Take a term structure with 8 points
2. Use cubic spline interpolation to create continuous curve
3. Extract prices at arbitrary maturities (e.g., 45 days, 100 days)
4. Visualize discrete points vs smooth curve

**Hint:** Use `scipy.interpolate.CubicSpline`

In [ ]:
# YOUR CODE HERE
from scipy.interpolate import CubicSpline

# YOUR CODE: Interpolate forward curve

---

## Summary

### Key Takeaways

1. **Term structure** reveals market expectations and supply/demand balance
2. **Contango** = upward sloping curve, negative roll yield
3. **Backwardation** = downward sloping, positive roll yield
4. **Roll yield** can dominate commodity investment returns
5. **PCA factors** (level, slope, curvature) capture curve dynamics
6. **Term structure features** improve forecast models

### Practical Applications

- **Trading**: Exploit roll yield through calendar spreads
- **Risk management**: Forward curves reveal price expectations
- **Forecasting**: Term structure features as predictors
- **Storage decisions**: Compare spot vs forward prices

### What's Next

Next module: **State Space Models** - Kalman filtering for level and trend estimation

### Additional Resources

- Geman (2005): *Commodities and Commodity Derivatives*
- Gorton & Rouwenhorst (2006): "Facts and Fantasies about Commodity Futures"
- Erb & Harvey (2006): "The Strategic and Tactical Value of Commodity Futures"

---